# TPS-VITON Training on Kaggle GPU

This notebook trains the **TPS-VITON** model from [github.com/Mothieram/Tryon](https://github.com/Mothieram/Tryon) on a Kaggle GPU.

## How to use
1. **Settings → Accelerator**: pick a GPU. **T4 x2** is recommended — the training scripts auto-wrap models in `nn.DataParallel` when more than one CUDA device is visible (no extra config needed). P100 (single-GPU) works too.
2. **Settings → Internet**: ON (needed to clone the repo and pip-install).
3. **Add Data**: attach your VITON-HD dataset from the Kaggle interface (right sidebar → *+ Add Input*).
4. Update `DATASET_ROOT` in the *Paths* cell to point at the attached dataset.
5. Run all cells.

The dataset directory must contain `train/` and `test/` subfolders with the standard VITON-HD subdirectories (`image/`, `cloth/`, `cloth-mask/`, `agnostic-v3.2/`, `image-densepose/`, `image-parse-v3/`, `image-parse-agnostic-v3.2/`, `openpose_json/`).

## Multi-GPU notes (T4 x2)
- `nn.DataParallel` activates automatically when `torch.cuda.device_count() > 1` — nothing to configure.
- With 2 GPUs, consider bumping `BATCH_SIZE` (the *global* batch size) — e.g. `12` or `16` instead of `8` — to keep both GPUs well-fed.
- Checkpoints are saved unwrapped, so you can resume on 1 GPU or 2 GPUs interchangeably.
- DataParallel does **not** sync BatchNorm across GPUs (would require DDP). Fine with per-GPU batches ≥ 4.

## 1. Clone the repository

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL  = "https://github.com/Mothieram/Tryon.git"
REPO_DIR  = Path("/kaggle/working/Tryon")
PROJECT   = REPO_DIR / "tps_vton"

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print("cwd =", os.getcwd())
print("contents:", sorted(os.listdir(PROJECT)))

## 2. Install dependencies

Kaggle images already ship Torch + torchvision built against CUDA, so we skip those and only install the rest.

In [ ]:
!pip install -q lpips torchmetrics tensorboard PyYAML scipy scikit-image

In [ ]:
import torch
print("torch =", torch.__version__)
print("cuda  =", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 3. Paths â€” edit these for your Kaggle dataset

`DATASET_ROOT` should point to the directory that contains `train/` and `test/`.
On Kaggle, attached datasets live under `/kaggle/input/<dataset-slug>/`.

In [ ]:
# === EDIT ME =================================================================
# Path to the VITON-HD dataset folder that holds train/ and test/.
DATASET_ROOT = "/kaggle/input/viton-hd"   # e.g. /kaggle/input/viton-hd-dataset/VITON-HD

# Optional: override training hyperparams without editing config.yaml.
STAGE        = "gmm"          # "gmm" | "refinement" | "joint"
BATCH_SIZE   = 8              # P100 ~8, T4 ~6, drop further if OOM
NUM_WORKERS  = 2              # Kaggle has limited CPU; 2 is safe
GMM_EPOCHS   = 60
REFINE_EPOCHS = 40
JOINT_EPOCHS = 15
RESOLUTION   = [256, 192]     # phase-1 resolution
AMP_ENABLED  = True
SMOKE        = False          # True = run a tiny dry-pass

# Where logs and checkpoints get written. /kaggle/working is persisted as the notebook output.
RUN_DIR      = "/kaggle/working/runs"
CKPT_DIR     = "/kaggle/working/checkpoints"

# Optional: resume / staged checkpoints (set to None to skip)
RESUME_FROM       = None
GMM_CHECKPOINT    = None
REFINE_CHECKPOINT = None
# =============================================================================

assert Path(DATASET_ROOT).exists(), f"DATASET_ROOT not found: {DATASET_ROOT}"
print("Dataset root exists. Contents:", sorted(os.listdir(DATASET_ROOT))[:20])

## 4. Generate a Kaggle-specific config

We load the repo's `configs/config.yaml`, override the paths and a few settings, and write a new `configs/config_kaggle.yaml`. The training script reads this file.

In [ ]:
import yaml

BASE_CFG = PROJECT / "configs" / "config.yaml"
KAGGLE_CFG = PROJECT / "configs" / "config_kaggle.yaml"

with open(BASE_CFG, "r") as f:
    cfg = yaml.safe_load(f)

# --- data -------------------------------------------------------------------
cfg["data"]["root"]        = DATASET_ROOT
cfg["data"]["num_workers"] = NUM_WORKERS
cfg["data"]["resolution"]  = RESOLUTION

# --- training ---------------------------------------------------------------
cfg["training"]["stage"]         = STAGE
cfg["training"]["batch_size"]    = BATCH_SIZE
cfg["training"]["gmm_epochs"]    = GMM_EPOCHS
cfg["training"]["refine_epochs"] = REFINE_EPOCHS
cfg["training"]["joint_epochs"]  = JOINT_EPOCHS
cfg["training"]["device"]        = "cuda"
cfg["training"]["amp"]["enabled"] = AMP_ENABLED

# --- logging ----------------------------------------------------------------
cfg["logging"]["log_dir"]  = RUN_DIR
cfg["logging"]["ckpt_dir"] = CKPT_DIR

Path(RUN_DIR).mkdir(parents=True, exist_ok=True)
Path(CKPT_DIR).mkdir(parents=True, exist_ok=True)

with open(KAGGLE_CFG, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(f"Wrote {KAGGLE_CFG}")
print(yaml.safe_dump({"data": cfg["data"], "training": cfg["training"], "logging": cfg["logging"]}, sort_keys=False))

## 5. (Optional) Verify dataset layout & build val split

Runs the repo's `prepare_data.py` to sanity-check VITON-HD subdirectories and create `val_pairs.txt`. Skip if you already have it.

In [ ]:
!python scripts/prepare_data.py --config configs/config_kaggle.yaml || echo "prepare_data finished with non-zero exit (often safe to ignore if pairs files already exist)"

## 6. (Optional) Smoke test

Quick sanity check that the model + data + losses all wire up before launching a long run.

In [ ]:
# Uncomment to run a smoke pass
# !python train.py --config configs/config_kaggle.yaml --stage gmm --smoke

## 7. Train

The command below dispatches to the right stage based on `STAGE`.

Kaggle GPU sessions cap at ~9â€“12h. For long runs, save checkpoints to `/kaggle/working/checkpoints` (already configured) so you can resume in a follow-up session by setting `RESUME_FROM`.

In [ ]:
cmd = [
    "python", "train.py",
    "--config", "configs/config_kaggle.yaml",
    "--stage", STAGE,
]
if RESUME_FROM:       cmd += ["--resume", RESUME_FROM]
if GMM_CHECKPOINT:    cmd += ["--gmm_checkpoint", GMM_CHECKPOINT]
if REFINE_CHECKPOINT: cmd += ["--refine_checkpoint", REFINE_CHECKPOINT]
if SMOKE:             cmd += ["--smoke"]

print("Launching:", " ".join(cmd))
subprocess.run(cmd, check=True)

## 8. Inspect outputs

Checkpoints and TensorBoard logs persist in `/kaggle/working/` and will be downloadable from the notebook output.

In [ ]:
!ls -lah /kaggle/working/checkpoints || true
!ls -lah /kaggle/working/runs || true

### Resuming a run
Set `RESUME_FROM = "/kaggle/working/checkpoints/<file>.pth"` in the *Paths* cell, re-run from cell 4 onward.

### Switching stages
After GMM training finishes, set `STAGE = "refinement"` and `GMM_CHECKPOINT = "/kaggle/working/checkpoints/best_lpips.pth"`. For `joint`, set both `GMM_CHECKPOINT` and `REFINE_CHECKPOINT`.